# Check the flawed copies

This prints each clean claim next to its flawed copy and the evidence.
The notes and answer key are private; do not send this notebook to the agents.
Nothing here writes reports or calls a model. Run all cells to check all pairs listed in `private/pairs.json`.

Reports are grouped by packet in `public/reports/`. Each folder has `clean.json`
and numbered error copies. Filenames and the notes below are for us, not the model.


In [ ]:
import sys
from pathlib import Path
from pprint import pprint

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/benchmark/private/pairs.json").is_file())
sys.path.insert(0, str(repo))
from src.benchmark.prepare import read, load_agent_input, check_report
from src.benchmark.create_flawed_reports import make_report
from src.benchmark.schemas import Packet, Report

folder = repo / "data/benchmark"
pairs = read(folder / "private/pairs.json")["pairs"]


In [ ]:
for pair in pairs:
    packet_path = folder / "public/evidence_packets" / (pair["packet_id"] + ".json")
    clean = load_agent_input(packet_path, folder / "public" / pair["clean_report_path"], check_clean=False)
    flawed = load_agent_input(packet_path, folder / "public" / pair["flawed_report_path"], check_clean=False)
    print("\n", pair["pair_id"], pair["packet_id"], pair["defect_type"])
    print("Error index:", pair["error_index"])
    print("Derived from:", pair["clean_report_path"])
    print("Flawed copy:", pair["flawed_report_path"])
    print("Same evidence:", clean["evidence_packet"] == flawed["evidence_packet"])
    print("Matches recorded edits:", make_report(clean["initial_report"], pair) == flawed["initial_report"])
    for change in pair["changes"]:
        print("\nChanged:", change["path"])
        print("BEFORE:")
        pprint(change["before"], sort_dicts=False)
        print("AFTER:")
        pprint(change["after"], sort_dicts=False)
    for evidence in clean["evidence_packet"]["evidence"]:
        if evidence["evidence_id"] in pair["evidence_ids"]:
            print("\nEVIDENCE:", evidence["evidence_id"], evidence["text"])
    claim = next((c for c in flawed["initial_report"]["claims"]
                  if c["claim_id"] == pair["target_claim_id"]), None)
    if claim is None and pair["target_claim_id"] is not None:
        claim = next(c for c in clean["initial_report"]["claims"]
                     if c["claim_id"] == pair["target_claim_id"])
    if pair["target_claim_id"] is None:
        print("Target: rationale")
    spec = claim["check_spec"] if claim else None
    print("\nReferenced facts:")
    if spec is None:
        print("No structured check_spec for this claim; review the evidence above.")
    else:
        ids = ([spec.get("fact_id")] if spec["kind"] == "numeric"
               else [spec.get("from_fact_id"), spec.get("to_fact_id")])
        pprint([f for f in clean["evidence_packet"]["facts"] if f["fact_id"] in ids])
    print("\nExpected:", pair["expected"])
    print("Injected:", pair["injected"])
    print("Reason:", pair["explanation"])
    print("Preparation checks (not experimental results):")
    pprint(check_report(Packet.model_validate(flawed["evidence_packet"]), Report.model_validate(flawed["initial_report"])))


## Notes

- Apple wrong number: FY2025 net sales changed from 416,161 to 426,161 million. The new 8.98% is calculated from the wrong amount, so the mistake is the source amount.
- Apple period: the FY2024-FY2025 comparison is labelled FY2023-FY2024. The amounts and 6.43% stay the same.
- Apple arithmetic: 6.43% becomes 16.43%; both source amounts stay the same.
- Microsoft wrong number: FY2025 revenue changed from 281,724 to 291,724 million. The corresponding growth is 19.01%.
- Microsoft period: the FY2024-FY2025 revenue comparison is labelled FY2023-FY2024. Operating income still refers to its original period.
- Microsoft arithmetic: 14.93% becomes 24.93%; both source amounts stay the same.
- Repeated assertions in the rationale follow the same intended error. All other claims, citations, recommendations and evidence stay unchanged.
- The period errors retain the clean source-fact IDs. The checker now identifies that the claimed periods differ from those facts, rather than falling back to a missing-period review flag.
- These are deliberately edited copies. They do not measure how often the Analyst makes these errors naturally or whether another agent will accept them.

Decision: the six copies contain the intended errors. Ready for paired experiments;
no confidence variants or experiment runs yet.
